In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS retailpulse.silver;

In [0]:
%sql
SHOW SCHEMAS IN retailpulse;

In [0]:
%sql
DESCRIBE retailpulse.bronze.customers;

In [0]:

from pyspark.sql.types import StructType, StructField, StringType, TimestampType
from pyspark.sql import functions as F


# ============================================================
# 1. CONFIGURATION
# ============================================================

source_table = "retailpulse.bronze.customers"
target_table = "retailpulse.silver.customers"
rejected_table = "retailpulse.silver.customers_rejected"

primary_key_column = "customer_id"


# ============================================================
# 2. DEFINE CUSTOMER SCHEMA
# ============================================================

customer_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("customer_unique_id", StringType(), False),
    StructField("customer_zip_code_prefix", StringType(), True),
    StructField("customer_city", StringType(), True),
    StructField("customer_state", StringType(), False),
    StructField("source_file", StringType(), True),
    StructField("ingested_at", TimestampType(), True),
    StructField("run_id", StringType(), True)
])


# ============================================================
# 3. READ BRONZE DATA
# ============================================================

customers_df = spark.read.table(source_table)

customers_df.limit(10).display()
customers_df.printSchema()

total_records_count = customers_df.count()

print("Total Bronze records:", total_records_count)


# ============================================================
# 4. STRING STANDARDIZATION
# ============================================================

string_columns = [
    field.name
    for field in customers_df.schema.fields
    if isinstance(field.dataType, StringType)
]

# Remove leading and trailing spaces
for column_name in string_columns:
    customers_df = customers_df.withColumn(
        column_name,
        F.trim(F.col(column_name))
    )

# Standardize city and state
customers_df = customers_df.withColumns({
    "customer_city": F.initcap(F.col("customer_city")),
    "customer_state": F.upper(F.col("customer_state"))
})

customers_df.limit(10).display()


# ============================================================
# 5. DATA QUALITY PROFILING
# ============================================================

# NULL count for every column
null_count_expressions = [
    F.coalesce(
        F.sum(
            F.when(F.col(column_name).isNull(), 1).otherwise(0)
        ),
        F.lit(0)
    ).alias(column_name)
    for column_name in customers_df.columns
]

null_counts_dict = (
    customers_df.agg(*null_count_expressions).first().asDict()
)


# Empty-string count for string columns
empty_string_expressions = [
    F.coalesce(
        F.sum(
            F.when(F.col(column_name) == "", 1).otherwise(0)
        ),
        F.lit(0)
    ).alias(column_name)
    for column_name in string_columns
]

empty_string_counts_dict = (
    customers_df.agg(*empty_string_expressions).first().asDict()
)

print("NULL COUNTS:", null_counts_dict)
print("EMPTY STRING COUNTS:", empty_string_counts_dict)


# ============================================================
# 6. MANDATORY FIELD VALIDATION
# ============================================================

mandatory_columns = [
    "customer_id",
    "customer_unique_id",
    "customer_state"
]

invalid_records_df = customers_df.filter(
    (F.col("customer_id").isNull()) |
    (F.col("customer_id") == "") |
    (F.col("customer_unique_id").isNull()) |
    (F.col("customer_unique_id") == "") |
    (F.col("customer_state").isNull()) |
    (F.col("customer_state") == "")
)

valid_records_df = customers_df.filter(
    (F.col("customer_id").isNotNull()) &
    (F.col("customer_id") != "") &
    (F.col("customer_unique_id").isNotNull()) &
    (F.col("customer_unique_id") != "") &
    (F.col("customer_state").isNotNull()) &
    (F.col("customer_state") != "")
)


# ============================================================
# 7. RECORD COUNTS AND DATA RECONCILIATION
# ============================================================

invalid_records_count = invalid_records_df.count()
valid_records_count = valid_records_df.count()

reconciled_count = valid_records_count + invalid_records_count

print("Bronze records:", total_records_count)
print("Valid records:", valid_records_count)
print("Invalid records:", invalid_records_count)
print("Reconciled records:", reconciled_count)

if total_records_count == reconciled_count:
    print("RECONCILIATION PASSED")
else:
    print("RECONCILIATION FAILED")


# ============================================================
# 8. PRIMARY KEY DUPLICATE ANALYSIS
# ============================================================

duplicate_customer_ids_df = (
    valid_records_df
    .groupBy(primary_key_column)
    .count()
    .filter(F.col("count") > 1)
)

# Number of duplicate customer ID groups
duplicate_customer_id_group_count = duplicate_customer_ids_df.count()

# Total records belonging to duplicate groups
duplicate_customer_record_count = (
    duplicate_customer_ids_df
    .agg(F.sum("count").alias("duplicate_record_count"))
    .first()["duplicate_record_count"] or 0
)

# Maximum occurrences of a single customer ID
max_customer_id_occurrence = (
    duplicate_customer_ids_df
    .agg(F.coalesce(F.max("count"), F.lit(0)).alias("max_occurrence"))
    .first()["max_occurrence"]
)

print("Duplicate customer ID groups:", duplicate_customer_id_group_count)
print("Records in duplicate groups:", duplicate_customer_record_count)
print("Maximum customer ID occurrence:", max_customer_id_occurrence)


# ============================================================
# 9. BUSINESS-LEVEL EXACT DUPLICATE ANALYSIS
# ============================================================

business_columns = [
    "customer_id",
    "customer_unique_id",
    "customer_zip_code_prefix",
    "customer_city",
    "customer_state"
]

exact_duplicates_df = (
    valid_records_df
    .groupBy(*business_columns)
    .count()
    .filter(F.col("count") > 1)
)

business_duplicate_metrics = exact_duplicates_df.agg(
    F.count("*").alias("duplicate_group_count"),
    F.coalesce(
        F.sum("count"), F.lit(0)
    ).alias("duplicate_record_count"),
    F.coalesce(
        F.sum(F.col("count") - 1), F.lit(0)
    ).alias("duplicate_excess_count")
).first()

exact_duplicate_business_group_count = (
    business_duplicate_metrics["duplicate_group_count"]
)

exact_duplicate_business_record_count = (
    business_duplicate_metrics["duplicate_record_count"]
)

exact_duplicate_business_excess_count = (
    business_duplicate_metrics["duplicate_excess_count"]
)


# ============================================================
# 10. FULL-RECORD EXACT DUPLICATE ANALYSIS
# ============================================================

exact_duplicate_full_records_df = (
    valid_records_df
    .groupBy(*valid_records_df.columns)
    .count()
    .filter(F.col("count") > 1)
)

full_records_duplicate_metrics = exact_duplicate_full_records_df.agg(
    F.count("*").alias("duplicate_group_count"),
    F.coalesce(
        F.sum("count"), F.lit(0)
    ).alias("duplicate_record_count"),
    F.coalesce(
        F.sum(F.col("count") - 1), F.lit(0)
    ).alias("duplicate_excess_count")
).first()

exact_duplicate_full_group_count = (
    full_records_duplicate_metrics["duplicate_group_count"]
)

exact_duplicate_full_record_count = (
    full_records_duplicate_metrics["duplicate_record_count"]
)

exact_duplicate_full_excess_count = (
    full_records_duplicate_metrics["duplicate_excess_count"]
)



# ============================================================
# 11. DUPLICATE ANALYSIS SUMMARY
# ============================================================

print("\n========== PRIMARY KEY DUPLICATES ==========")
print("Duplicate ID groups:", duplicate_customer_id_group_count)
print("Duplicate records:", duplicate_customer_record_count)
print("Maximum ID occurrence:", max_customer_id_occurrence)

print("\n========== BUSINESS-LEVEL DUPLICATES ==========")
print("Duplicate groups:", exact_duplicate_business_group_count)
print("Duplicate records:", exact_duplicate_business_record_count)
print("Excess duplicates:", exact_duplicate_business_excess_count)

print("\n========== FULL-RECORD DUPLICATES ==========")
print("Duplicate groups:", exact_duplicate_full_group_count)
print("Duplicate records:", exact_duplicate_full_record_count)
print("Excess duplicates:", exact_duplicate_full_excess_count)


# ============================================================
# 12. CUSTOMER STATE VALIDATION
# ============================================================

# Step 1: Get distinct customer states from actual data

distinct_state_df = valid_records_df.select(
    'customer_state'
).distinct()


# Step 2: Count customers belonging to each state

each_customer_state_count = valid_records_df.groupBy(
    'customer_state'
).count()

print("\n========== CUSTOMER STATE DISTRIBUTION ==========")
display(each_customer_state_count)


# Step 3: Define expected valid states

states = [
    'AC', 'AL', 'AP', 'AM', 'BA', 'CE', 'DF', 'ES',
    'GO', 'MA', 'MT', 'MS', 'MG', 'PA', 'PB', 'PR',
    'PE', 'PI', 'RJ', 'RN', 'RS', 'RO', 'RR', 'SC',
    'SP', 'SE', 'TO'
]


# Step 4: Convert expected states into a Spark DataFrame

state_data = []

for state in states:
    state_data.append((state,))

expected_states_df = spark.createDataFrame(
    state_data,
    ['customer_state']
)


# Step 5: Identify invalid states using Left Anti Join

invalid_state_df = distinct_state_df.join(
    expected_states_df,
    on='customer_state',
    how='left_anti'
)


# Step 6: Count distinct invalid states

invalid_states_count = invalid_state_df.count()


# Step 7: Display invalid states

print("\n========== INVALID CUSTOMER STATES ==========")
display(invalid_state_df)


# Step 8: Identify customer records having invalid states

invalid_customer_state_df = valid_records_df.join(
    invalid_state_df,
    on='customer_state',
    how='inner'
)


# Step 9: Count invalid customer records

invalid_customer_state_record_count = invalid_customer_state_df.count()


# Step 10: Calculate valid customer records

total_customer_records = valid_records_df.count()

valid_customer_state_record_count = (
    total_customer_records - invalid_customer_state_record_count
)


# Step 11: Customer state validation summary

print("\n========== CUSTOMER STATE VALIDATION SUMMARY ==========")

print("Expected distinct states:", len(states))
print("Actual distinct states:", distinct_state_df.count())
print("Invalid distinct states:", invalid_states_count)

print("Total customer records:", total_customer_records)
print("Valid customer state records:", valid_customer_state_record_count)
print("Invalid customer state records:", invalid_customer_state_record_count)




